<a href="https://colab.research.google.com/github/Armin-HassanzadehHassanabad/Airline-satisfaction-kaggle/blob/main/airlinesatisfaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e10_path = kagglehub.competition_download('playground-series-s6e10')

print('Data source import complete.')


In [ ]:
from sklearn.model_selection import StratifiedKFold

def make_target_encoding(keys_fit, y_fit, keys_apply, smoothing=20):
    stats = pd.DataFrame({
        "key": np.asarray(keys_fit),
        "target": np.asarray(y_fit)
    }).groupby("key")["target"].agg(["mean", "count"])

    prior = np.mean(y_fit)

    encoding = (
        stats["mean"] * stats["count"] + prior * smoothing
    ) / (stats["count"] + smoothing)

    return pd.Series(np.asarray(keys_apply)).map(encoding).fillna(prior).to_numpy()

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')

y = train['satisfaction'].astype(int)
X = train.drop(columns=['id', 'satisfaction'])
X_test = test.drop(columns=['id'])

X['Is_Delayed_30']      = (X['Departure Delay in Minutes'] > 30).astype(int)
X_test['Is_Delayed_30'] = (X_test['Departure Delay in Minutes'] > 30).astype(int)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Target rate:", y.mean())

X shape: (699635, 22)
X_test shape: (299844, 22)
Target rate: 0.44357272006117476


In [ ]:
def build_features(df):
    out = df.copy()
    d = out['Flight Distance'].astype(int)
    out['Dist_div10']   = d // 10
    out['Dist_mod1000'] = d % 1000
    out['Dist_mod100']  = d % 100
    out['Dist_mod10']   = d % 10
    out['Dist_x_Class'] = d.astype(str) + "_" + out['Class'].astype(str)
    out['Age_x_Class']  = out['Age'].astype(str) + "_" + out['Class'].astype(str)
    return out

X_fe = build_features(X)
X_test_fe = build_features(X_test)

In [ ]:
num_cols_for_profile = [
    'Age', 'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]
num_cols_for_profile = [c for c in num_cols_for_profile if c in X_fe.columns]

profile = X_fe.groupby('Flight Distance')[num_cols_for_profile].mean()
profile.columns = [f'RouteProfile_{c}' for c in profile.columns]

X_fe2 = X_fe.merge(profile, left_on='Flight Distance', right_index=True, how='left')
X_test_fe2 = X_test_fe.merge(profile, left_on='Flight Distance', right_index=True, how='left')

for c in profile.columns:
    X_fe2[c] = X_fe2[c].fillna(X_fe2[c].median())
    X_test_fe2[c] = X_test_fe2[c].fillna(X_fe2[c].median())

print("X_fe2 shape:", X_fe2.shape)
print("X_test_fe2 shape:", X_test_fe2.shape)

X_fe2 shape: (699635, 42)
X_test_fe2 shape: (299844, 42)


In [ ]:
from sklearn.model_selection import StratifiedKFold

te_cols = ['Flight Distance', 'Dist_div10', 'Dist_mod1000', 'Age_x_Class', 'Dist_x_Class']

cat_cols_final = X_fe2.select_dtypes(include='object').columns.tolist()
cat_cols_final = [c for c in cat_cols_final if c not in te_cols]

# 5-Fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("te_cols:", te_cols)
print("cat_cols_final:", cat_cols_final)

te_cols: ['Flight Distance', 'Dist_div10', 'Dist_mod1000', 'Age_x_Class', 'Dist_x_Class']
cat_cols_final: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


In [ ]:
import xgboost as xgb

# One-hot encode categoricals
X_xgb2 = pd.get_dummies(X_fe2.drop(columns=te_cols), columns=cat_cols_final, drop_first=False)
X_test_xgb2 = pd.get_dummies(X_test_fe2.drop(columns=te_cols), columns=cat_cols_final, drop_first=False)
X_test_xgb2 = X_test_xgb2.reindex(columns=X_xgb2.columns, fill_value=0)

params_xgb = dict(
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_lambda=1.0,
    random_state=42,
    n_estimators=3000,
    early_stopping_rounds=100,
    verbosity=0
)

print("X_xgb2 shape:", X_xgb2.shape)
print("X_test_xgb2 shape:", X_test_xgb2.shape)

X_xgb2 shape: (699635, 42)
X_test_xgb2 shape: (299844, 42)


In [ ]:
# Preserve the original missing-value information
X_fe3 = X_fe2.copy()
X_test_fe3 = X_test_fe2.copy()

X_fe3["Arrival_Delay_Missing"] = (
    X_fe3["Arrival Delay in Minutes"].isna().astype("int8")
)

X_test_fe3["Arrival_Delay_Missing"] = (
    X_test_fe3["Arrival Delay in Minutes"].isna().astype("int8")
)

print("Train shape:", X_fe3.shape)
print("Test shape:", X_test_fe3.shape)
print(
    "Missing indicator counts:",
    X_fe3["Arrival_Delay_Missing"].sum(),
    X_test_fe3["Arrival_Delay_Missing"].sum()
)

Train shape: (699635, 43)
Test shape: (299844, 43)
Missing indicator counts: 292 130


In [ ]:
import time
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Reuse the same feature matrix used by the corrected XGBoost model
X_exp = X_xgb2.copy()
X_exp_test = X_test_xgb2.copy()

y_exp = y.reset_index(drop=True)
X_exp = X_exp.reset_index(drop=True)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_exp,
    y_exp,
    test_size=0.20,
    random_state=42,
    stratify=y_exp
)

model_xgb_exp = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    device="cuda",
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_estimators=2000,
    early_stopping_rounds=100,
    random_state=42,
    verbosity=0
)

start = time.time()

model_xgb_exp.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=200
)

pred_valid = model_xgb_exp.predict_proba(
    X_valid,
    iteration_range=(0, model_xgb_exp.best_iteration + 1)
)[:, 1]

print(f"Validation AUC: {roc_auc_score(y_valid, pred_valid):.6f}")
print(f"Best iteration: {model_xgb_exp.best_iteration}")
print(f"Runtime: {(time.time() - start) / 60:.2f} minutes")

[0]	validation_0-auc:0.95011
[200]	validation_0-auc:0.95801
[400]	validation_0-auc:0.95917
[600]	validation_0-auc:0.95944
[790]	validation_0-auc:0.95945
Validation AUC: 0.959457
Best iteration: 690
Runtime: 1.99 minutes


In [ ]:

import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --------------------------------------------------
# 1. Feature engineering
# --------------------------------------------------
A = X_fe3.copy().reset_index(drop=True)
B = X_test_fe3.copy().reset_index(drop=True)

service_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]
service_cols = [c for c in service_cols if c in A.columns]

def add_advanced_features(df):
    df = df.copy()

    # Service-rating profile
    s = df[service_cols].astype(float)
    df["Service_Mean"] = s.mean(axis=1)
    df["Service_Sum"] = s.sum(axis=1)
    df["Service_Std"] = s.std(axis=1)
    df["Service_Min"] = s.min(axis=1)
    df["Service_Max"] = s.max(axis=1)
    df["Service_Low_Count"] = (s <= 2).sum(axis=1)
    df["Service_High_Count"] = (s >= 4).sum(axis=1)

    # Selected service interactions
    if "Inflight wifi service" in df and "Online boarding" in df:
        df["Wifi_x_Boarding"] = (
            df["Inflight wifi service"] * df["Online boarding"]
        )
        df["Wifi_Boarding_Diff"] = (
            df["Inflight wifi service"] - df["Online boarding"]
        )

    if "Seat comfort" in df and "Inflight entertainment" in df:
        df["Seat_x_Entertainment"] = (
            df["Seat comfort"] * df["Inflight entertainment"]
        )

    # Delay relationships
    dep = df["Departure Delay in Minutes"].clip(lower=0)
    arr = df["Arrival Delay in Minutes"].clip(lower=0)

    df["Delay_Total"] = dep + arr
    df["Delay_Difference"] = arr - dep
    df["Delay_Log_Dep"] = np.log1p(dep)
    df["Delay_Log_Arr"] = np.log1p(arr)
    df["Delay_Both_15"] = ((dep > 15) & (arr > 15)).astype("int8")
    df["Delay_Both_30"] = ((dep > 30) & (arr > 30)).astype("int8")
    df["Arrival_Less_Than_Departure"] = (arr < dep).astype("int8")

    # Age and distance bands
    df["Age_Band"] = pd.cut(
        df["Age"],
        bins=[-1, 18, 25, 35, 45, 55, 65, 120],
        labels=False
    ).fillna(-1).astype(str)

    df["Distance_Band"] = pd.cut(
        df["Flight Distance"],
        bins=[-1, 500, 1000, 1500, 2000, 3000, 10000],
        labels=False
    ).fillna(-1).astype(str)

    # Categorical interactions
    df["Travel_Class"] = (
        df["Type of Travel"].astype(str) + "_" +
        df["Class"].astype(str)
    )
    df["Customer_Travel"] = (
        df["Customer Type"].astype(str) + "_" +
        df["Type of Travel"].astype(str)
    )
    df["Age_x_Class"] = (
        df["Age"].astype(str) + "_" + df["Class"].astype(str)
    )
    df["Dist_x_Class"] = (
        df["Flight Distance"].astype(str) + "_" +
        df["Class"].astype(str)
    )

    return df

A = add_advanced_features(A)
B = add_advanced_features(B)

# Target encoding keys: learned from training labels inside each fold
te_cols = [
    "Flight Distance",
    "Dist_div10",
    "Dist_mod1000",
    "Age_x_Class",
    "Dist_x_Class",
    "Distance_Band",
    "Travel_Class",
    "Customer_Travel"
]
te_cols = [c for c in te_cols if c in A.columns]

# One-hot encode all remaining categorical features
cat_cols = A.drop(columns=te_cols).select_dtypes(
    include=["object", "category"]
).columns.tolist()

XA = pd.get_dummies(
    A.drop(columns=te_cols),
    columns=cat_cols,
    dtype=np.uint8
)
XB = pd.get_dummies(
    B.drop(columns=te_cols),
    columns=cat_cols,
    dtype=np.uint8
)
XB = XB.reindex(columns=XA.columns, fill_value=0)

XA = XA.reset_index(drop=True)
XB = XB.reset_index(drop=True)
y_array = np.asarray(y, dtype=np.int8)

# --------------------------------------------------
# 2. Smoothed target encoding
# --------------------------------------------------
def make_te(keys_fit, y_fit, keys_apply, smoothing=30):
    k = pd.Series(np.asarray(keys_fit)).reset_index(drop=True)
    yy = pd.Series(np.asarray(y_fit)).reset_index(drop=True)
    ka = pd.Series(np.asarray(keys_apply))

    prior = yy.mean()
    stats = pd.DataFrame({"key": k, "target": yy}).groupby(
        "key", dropna=False
    )["target"].agg(["mean", "count"])

    mapping = (
        stats["mean"] * stats["count"] + prior * smoothing
    ) / (stats["count"] + smoothing)

    return ka.map(mapping).fillna(prior).to_numpy(dtype=np.float32)

# --------------------------------------------------
# 3. Five-fold XGBoost with leakage-safe encodings
# --------------------------------------------------
outer_cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

oof_xgb5 = np.zeros(len(A), dtype=np.float32)
pred_xgb5 = np.zeros(len(B), dtype=np.float32)
fold_scores = []
start = time.time()

for fold, (tr_idx, va_idx) in enumerate(
    outer_cv.split(XA, y_array), 1
):
    print(f"\n========== Fold {fold}/5 ==========")

    y_tr = y_array[tr_idx]
    y_va = y_array[va_idx]

    X_tr = XA.iloc[tr_idx].copy()
    X_va = XA.iloc[va_idx].copy()
    X_te = XB.copy()

    # OOF encodings for outer-training rows
    inner_cv = StratifiedKFold(
        n_splits=4, shuffle=True, random_state=100 + fold
    )

    for col in te_cols:
        enc_train = np.zeros(len(tr_idx), dtype=np.float32)

        keys_outer_train = A.iloc[tr_idx][col].to_numpy()
        keys_valid = A.iloc[va_idx][col].to_numpy()
        keys_test = B[col].to_numpy()

        for inner_tr, inner_va in inner_cv.split(
            np.zeros(len(tr_idx)), y_tr
        ):
            enc_train[inner_va] = make_te(
                keys_outer_train[inner_tr],
                y_tr[inner_tr],
                keys_outer_train[inner_va]
            )

        # Validation and test encoded using only outer-training data
        enc_valid = make_te(
            keys_outer_train, y_tr, keys_valid
        )
        enc_test = make_te(
            keys_outer_train, y_tr, keys_test
        )

        X_tr[f"TE_{col}"] = enc_train
        X_va[f"TE_{col}"] = enc_valid
        X_te[f"TE_{col}"] = enc_test

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cuda",
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        n_estimators=3000,
        early_stopping_rounds=100,
        random_state=42 + fold,
        verbosity=0,
        n_jobs=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=200
    )

    p_va = model.predict_proba(
        X_va,
        iteration_range=(0, model.best_iteration + 1)
    )[:, 1]

    p_te = model.predict_proba(
        X_te,
        iteration_range=(0, model.best_iteration + 1)
    )[:, 1]

    oof_xgb5[va_idx] = p_va
    pred_xgb5 += p_te / 5

    score = roc_auc_score(y_va, p_va)
    fold_scores.append(score)

    print(
        f"Fold {fold} AUC: {score:.6f} | "
        f"Best iteration: {model.best_iteration}"
    )

    del X_tr, X_va, X_te, model

print("\n========== FINAL RESULTS ==========")
print("Fold AUCs:", [round(s, 6) for s in fold_scores])
print(f"Mean fold AUC: {np.mean(fold_scores):.6f}")
print(f"OOF AUC: {roc_auc_score(y_array, oof_xgb5):.6f}")
print(f"Runtime: {(time.time() - start) / 60:.1f} minutes")

np.save("/kaggle/working/oof_xgb5.npy", oof_xgb5)
np.save("/kaggle/working/pred_xgb5.npy", pred_xgb5)



========== Fold 1/5 ==========
[0]	validation_0-auc:0.95047
[200]	validation_0-auc:0.95899
[400]	validation_0-auc:0.96008
[600]	validation_0-auc:0.96034
[800]	validation_0-auc:0.96045
[1000]	validation_0-auc:0.96047
[1138]	validation_0-auc:0.96047
Fold 1 AUC: 0.960484 | Best iteration: 1038

========== Fold 2/5 ==========
[0]	validation_0-auc:0.94402
[200]	validation_0-auc:0.95760
[400]	validation_0-auc:0.95881
[600]	validation_0-auc:0.95916
[800]	validation_0-auc:0.95929
[1000]	validation_0-auc:0.95934
[1200]	validation_0-auc:0.95937
[1304]	validation_0-auc:0.95935
Fold 2 AUC: 0.959382 | Best iteration: 1204

========== Fold 3/5 ==========
[0]	validation_0-auc:0.94794
[200]	validation_0-auc:0.95930
[400]	validation_0-auc:0.96056
[600]	validation_0-auc:0.96094
[800]	validation_0-auc:0.96108
[1000]	validation_0-auc:0.96111
[1038]	validation_0-auc:0.96110
Fold 3 AUC: 0.961116 | Best iteration: 938

========== Fold 4/5 ==========
[0]	validation_0-auc:0.95086
[200]	validation_0-auc:0.9590

In [ ]:
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Use original engineered features; CatBoost handles categories natively
A = X_fe3.copy().reset_index(drop=True)
B = X_test_fe3.copy().reset_index(drop=True)

cat_cols = A.select_dtypes(include=["object", "category"]).columns.tolist()

for c in cat_cols:
    A[c] = A[c].fillna("__MISSING__").astype(str)
    B[c] = B[c].fillna("__MISSING__").astype(str)

# CatBoost accepts categorical features by column index
cat_indices = [A.columns.get_loc(c) for c in cat_cols]

y_arr = np.asarray(y, dtype=np.int8)
oof_cat = np.zeros(len(A), dtype=np.float32)
pred_cat = np.zeros(len(B), dtype=np.float32)
scores = []
start = time.time()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(cv.split(A, y_arr), 1):
    print(f"\n========== Fold {fold}/5 ==========")

    model = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.04,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        l2_leaf_reg=5,
        random_seed=42 + fold,
        thread_count=-1,
        verbose=200,
        allow_writing_files=False
    )

    model.fit(
        A.iloc[tr_idx], y_arr[tr_idx],
        cat_features=cat_indices,
        eval_set=(A.iloc[va_idx], y_arr[va_idx]),
        early_stopping_rounds=150,
        verbose=200
    )

    p_va = model.predict_proba(A.iloc[va_idx])[:, 1]
    p_te = model.predict_proba(B)[:, 1]

    oof_cat[va_idx] = p_va
    pred_cat += p_te / 5

    score = roc_auc_score(y_arr[va_idx], p_va)
    scores.append(score)

    print(f"Fold {fold} AUC: {score:.6f}")
    print(f"Best iteration: {model.best_iteration_}")

    # Save progress after each fold
    np.save("/kaggle/working/oof_cat.npy", oof_cat)
    np.save("/kaggle/working/pred_cat.npy", pred_cat)

    del model

print("\nFold AUCs:", [round(s, 6) for s in scores])
print(f"Mean fold AUC: {np.mean(scores):.6f}")
print(f"OOF AUC: {roc_auc_score(y_arr, oof_cat):.6f}")
print(f"Runtime: {(time.time() - start) / 60:.1f} minutes")


========== Fold 1/5 ==========
0:	test: 0.9430371	best: 0.9430371 (0)	total: 609ms	remaining: 25m 21s
200:	test: 0.9585183	best: 0.9585183 (200)	total: 1m 32s	remaining: 17m 41s
400:	test: 0.9596191	best: 0.9596191 (400)	total: 3m 5s	remaining: 16m 12s
600:	test: 0.9599972	best: 0.9599972 (600)	total: 4m 34s	remaining: 14m 27s
800:	test: 0.9602659	best: 0.9602659 (800)	total: 6m 6s	remaining: 12m 58s
1000:	test: 0.9604156	best: 0.9604157 (999)	total: 7m 39s	remaining: 11m 27s
1200:	test: 0.9605081	best: 0.9605170 (1189)	total: 9m 10s	remaining: 9m 55s
1400:	test: 0.9605971	best: 0.9605987 (1390)	total: 10m 43s	remaining: 8m 24s
1600:	test: 0.9606426	best: 0.9606426 (1600)	total: 12m 16s	remaining: 6m 53s
1800:	test: 0.9606547	best: 0.9606571 (1693)	total: 13m 49s	remaining: 5m 21s
2000:	test: 0.9606820	best: 0.9606820 (2000)	total: 15m 22s	remaining: 3m 49s
2200:	test: 0.9606966	best: 0.9606966 (2200)	total: 16m 55s	remaining: 2m 17s
2400:	test: 0.9607119	best: 0.9607119 (2400)	total:

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

y_true = np.asarray(y, dtype=int)
oof_xgb = np.load("/kaggle/working/oof_xgb5.npy")
oof_cat = np.load("/kaggle/working/oof_cat.npy")

print("XGBoost:", roc_auc_score(y_true, oof_xgb))
print("CatBoost:", roc_auc_score(y_true, oof_cat))

for w in [0.2, 0.4, 0.5, 0.6, 0.8]:
    blend = (1 - w) * oof_xgb + w * oof_cat
    print(f"CatBoost weight={w:.1f}: AUC={roc_auc_score(y_true, blend):.6f}")

XGBoost: 0.9603367094022134
CatBoost: 0.9605376512714027
CatBoost weight=0.2: AUC=0.960572
CatBoost weight=0.4: AUC=0.960698
CatBoost weight=0.5: AUC=0.960725
CatBoost weight=0.6: AUC=0.960729
CatBoost weight=0.8: AUC=0.960675


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from pathlib import Path

# Load saved predictions
y_true = np.asarray(y, dtype=int)
oof_xgb = np.load("/kaggle/working/oof_xgb5.npy")
pred_xgb = np.load("/kaggle/working/pred_xgb5.npy")

oof_cat = np.load("/kaggle/working/oof_cat.npy")
pred_cat = np.load("/kaggle/working/pred_cat.npy")

# Find the best blend among a small, predefined set of weights
weights = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
results = []

for w in weights:
    oof_blend = (1 - w) * oof_xgb + w * oof_cat
    auc = roc_auc_score(y_true, oof_blend)
    results.append((w, auc))
    print(f"CatBoost weight={w:.1f} | OOF AUC={auc:.6f}")

best_w, best_auc = max(results, key=lambda z: z[1])

print(f"\nSelected CatBoost weight: {best_w:.1f}")
print(f"Selected OOF AUC: {best_auc:.6f}")

# Blend test predictions with the same weight
pred_blend = (1 - best_w) * pred_xgb + best_w * pred_cat

# Create submission using the competition's test IDs
submission = pd.DataFrame({
    "id": test["id"],
    "satisfaction": pred_blend
})

out_path = "/kaggle/working/submission_final.csv"
submission.to_csv(out_path, index=False)

print("\nSubmission saved:", out_path)
print("Shape:", submission.shape)
print(submission.head())
print("Missing predictions:", submission["satisfaction"].isna().sum())
print("Prediction range:",
      submission["satisfaction"].min(),
      submission["satisfaction"].max())


CatBoost weight=0.0 | OOF AUC=0.960337
CatBoost weight=0.2 | OOF AUC=0.960572
CatBoost weight=0.4 | OOF AUC=0.960698
CatBoost weight=0.5 | OOF AUC=0.960725
CatBoost weight=0.6 | OOF AUC=0.960729
CatBoost weight=0.8 | OOF AUC=0.960675
CatBoost weight=1.0 | OOF AUC=0.960538

Selected CatBoost weight: 0.6
Selected OOF AUC: 0.960729

Submission saved: /kaggle/working/submission_final.csv
Shape: (299844, 2)
       id  satisfaction
0  699635      0.259029
1  699636      0.839712
2  699637      0.907748
3  699638      0.064852
4  699639      0.070092
Missing predictions: 0
Prediction range: 0.002935989759862423 0.9982735514640808
